# Week 5 Lab：Stochastic Interpolants、曲率與 reflow

## 學習目標
- 比較 deterministic path 與帶 $\gamma_t z$ 的 stochastic interpolant。
- 看見 reflow 如何把解析 teacher 產生的 endpoint pair 拉成直線。
- 數值連結軌跡曲率積分與一步 Euler endpoint error。

> **誠實註記**：teacher 是下方明寫的解析旋轉 flow；reflow 是幾何 surrogate，沒有載入任何模型權重。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 1618
rng = np.random.default_rng(SEED)
centers = np.array([[-1.4, -0.8], [-1.2, 1.0], [1.3, -0.9], [1.4, 1.0]])

def sample_target(n):
    k = rng.integers(0, len(centers), size=n)
    return centers[k] + .18 * rng.standard_normal((n, 2))

x0 = rng.standard_normal((700, 2))
x1 = sample_target(700)
z = rng.standard_normal((700, 2))

def gamma(t, eta):
    return eta * np.sqrt(2 * t * (1 - t))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)
for ax, eta in zip(axes, [0.0, 0.35, 0.75]):
    t = .5
    xt = (1 - t) * x0 + t * x1 + gamma(t, eta) * z
    ax.scatter(xt[:, 0], xt[:, 1], s=5, alpha=.35)
    ax.set(title=rf'$\gamma$ strength = {eta}', xlim=(-3, 3), ylim=(-3, 3), aspect='equal')
plt.show()

In [ ]:
def rotate(x, angle):
    c, s = np.cos(angle), np.sin(angle)
    R = np.array([[c, -s], [s, c]])
    return np.asarray(x) @ R.T

shift = np.array([.65, -.2])
def teacher_path(x, t, twist=0.72):
    # Nonconstant angular speed creates a curved but exactly known flow.
    return rotate(x, twist * np.pi * t ** 2) + t * shift

def reflow_path(x, t, twist=0.72):
    end = teacher_path(x, 1.0, twist)
    return (1 - t) * x + t * end

seeds = rng.standard_normal((18, 2))
ts = np.linspace(0, 1, 100)
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for seed in seeds:
    curved = np.array([teacher_path(seed, t) for t in ts])
    straight = np.array([reflow_path(seed, t) for t in ts])
    axes[0].plot(curved[:, 0], curved[:, 1], alpha=.65)
    axes[1].plot(straight[:, 0], straight[:, 1], alpha=.65)
for ax, title in zip(axes, ['analytic teacher trajectories', 'reflow chords from teacher pairs']):
    ax.set(aspect='equal', title=title, xlim=(-3, 3), ylim=(-3, 3))
plt.show()

In [ ]:
def path_metrics(twist, n_paths=250, n_time=201):
    starts = rng.standard_normal((n_paths, 2))
    t = np.linspace(0, 1, n_time)
    paths = np.stack([teacher_path(starts, q, twist) for q in t])
    velocity = np.gradient(paths, t, axis=0, edge_order=2)
    accel = np.gradient(velocity, t, axis=0, edge_order=2)
    curvature_integral = np.trapezoid(np.linalg.norm(accel, axis=2), t, axis=0).mean()
    euler_one_step = starts + velocity[0]
    endpoint_error = np.linalg.norm(euler_one_step - paths[-1], axis=1).mean()
    return curvature_integral, endpoint_error

twists = np.linspace(.08, .95, 12)
metrics = np.array([path_metrics(w) for w in twists])
correlation = np.corrcoef(metrics.T)[0, 1]
print(f'correlation(curvature integral, one-step error) = {correlation:.3f}')
fig, ax = plt.subplots(figsize=(5.5, 4))
ax.scatter(metrics[:, 0], metrics[:, 1], c=twists, cmap='plasma', s=55)
for i in [0, len(twists)//2, -1]:
    ax.annotate(f'twist={twists[i]:.2f}', metrics[i], xytext=(5, 4), textcoords='offset points')
ax.set(xlabel='integrated acceleration surrogate', ylabel='one-step Euler endpoint error',
       title='Curvier teacher flow is harder to compress')
plt.show()

In [ ]:
seed = np.array([1.1, -.45])
teacher = np.array([teacher_path(seed, t) for t in ts])
reflow = np.array([reflow_path(seed, t) for t in ts])
teacher_speed = np.linalg.norm(np.gradient(teacher, ts, axis=0), axis=1)
reflow_speed = np.linalg.norm(np.gradient(reflow, ts, axis=0), axis=1)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ts, teacher_speed, label='teacher speed')
ax.plot(ts, reflow_speed, label='reflow chord speed')
ax.set(xlabel='t', ylabel='speed', title='Reflow makes the conditional target constant')
ax.legend()
plt.show()

## 讀者練習 / TODO
把 `teacher_path` 的角度從 $t^2$ 改成 $t$。預測並驗證：軌跡仍然彎，但速度是否更接近常數？曲率 surrogate 與一步誤差的關係會怎麼變？